# 👁️ Diabetic Retinopathy Severity Grading — Master Colab Pipeline

> **Project:** Public Portfolio Diabetic Retinopathy Grading with CORAL Ordinal Regression  
> **Target Hardware:** Google Colab T4 GPU (Free Tier)  
> **Workflow:** Phase 0 (Verification) → Phase 1 (CE Baseline) → Phase 2 (Focal Loss) → Phase 3 (CORAL Ordinal + W&B Sweep) → Phase 4 (Calibration) → Phase 5 (Grad-CAM) → Phase 6 (Messidor-2 Eval)

---

### ⚠️ Step 0: Ensure GPU is Enabled in Colab
Before running any cells, verify you are connected to a GPU runtime:
1. Click **Runtime** in the top menu.
2. Select **Change runtime type**.
3. Under **Hardware accelerator**, select **T4 GPU**.
4. Click **Save**.

In [ ]:
# Verify GPU allocation
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

---
## 📂 Step 1: Mount Google Drive & Set Up Project Directory
Mount your Google Drive where you have uploaded the project folder or dataset.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Set your project path here (adjust folder name if different on your Drive)
PROJECT_DIR = '/content/drive/MyDrive/dr-severity-grading'

if os.path.exists(PROJECT_DIR):
    %cd $PROJECT_DIR
    print(f"Changed working directory to: {os.getcwd()}")
else:
    print(f"Note: {PROJECT_DIR} not found. If running from local clone or root, check your path.")

---
## 📦 Step 2: Install Colab Dependencies & Package
Colab already has PyTorch, Torchvision, Scikit-learn, OpenCV, and Pandas.
We only need to install `timm`, `hydra-core`, `wandb`, `grad-cam`, `netcal`, and install our repo in editable mode.

In [ ]:
# Install dependencies from requirements-colab.txt
!pip install -q timm albumentations hydra-core omegaconf wandb grad-cam netcal huggingface_hub pytest
!pip install -q -e .
print("All dependencies successfully installed!")

---
## ✅ Step 3: Phase 0 Verification (Run PyTest Suite)
Run all unit tests to verify:
- Ben Graham preprocessing logic & circle-crop ROI
- Dataset loaders & class weight calculations
- CORAL ordinal loss numerical correctness, differentiability, & rank decoding
- Quadratic Weighted Kappa (QWK) & Expected Calibration Error (ECE)
- Grad-CAM hooks and activation overlays

In [ ]:
!pytest tests/ -v --tb=short

---
## ⚡ Step 4: Precompute Ben Graham Preprocessing (Offline Optimization)
> **Why Offline?** Per our `docs/lessons_learned.md`, running CLAHE + Gaussian blur on-the-fly causes CPU bottlenecks and starves the T4 GPU. Precomputing images to 380x380 increases GPU utilization to 95%+ and cuts training time per epoch from ~18 min to ~4.5 min.

In [ ]:
import os
import glob
import cv2
from tqdm import tqdm
from src.data.preprocessing import ben_graham_pipeline

def precompute_dataset(input_dir, output_dir, image_size=380):
    os.makedirs(output_dir, exist_ok=True)
    img_paths = glob.glob(os.path.join(input_dir, "*.[pP][nN][gG]")) + glob.glob(os.path.join(input_dir, "*.[jJ][pP][gG]"))
    print(f"Found {len(img_paths)} images in {input_dir}. Preprocessing...")
    
    for p in tqdm(img_paths):
        fname = os.path.basename(p)
        out_path = os.path.join(output_dir, os.path.splitext(fname)[0] + ".png")
        if os.path.exists(out_path):
            continue
        img = cv2.imread(p)
        if img is not None:
            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            processed = ben_graham_pipeline(rgb, image_size=image_size)
            cv2.imwrite(out_path, cv2.cvtColor(processed, cv2.COLOR_RGB2BGR))

# Uncomment and run once dataset is placed in data/aptos2019/train_images:
# precompute_dataset('data/aptos2019/train_images', 'data/aptos2019/preprocessed_380', image_size=380)

---
## 📊 Step 5: Weights & Biases Authentication
Log in to your W&B account to stream live training metrics, QWK curves, and sweep visualizations.

In [ ]:
import wandb
wandb.login()

---
## 🚀 Phase 1: Train Baseline (EfficientNet-B0 + Cross-Entropy)
Trains the baseline strawman with inverse-frequency class weights.

In [ ]:
# Run Phase 1 training
!python scripts/train.py loss=cross_entropy model=efficientnet_b0 training=baseline

---
## 🎯 Phase 2: Focal Loss Imbalance Ablation
Trains with Focal Loss (gamma=2.0) to isolate class imbalance handling.

In [ ]:
# Run Phase 2 training
!python scripts/train.py loss=focal model=efficientnet_b0 loss.gamma=2.0

---
## ⭐ Phase 3: CORAL Ordinal Regression Training & W&B Micro-Sweep
### Step 3A: Train Default CORAL Model

In [ ]:
# Run Phase 3 default CORAL training
!python scripts/train.py loss=coral model=efficientnet_b0 training=ordinal

### Step 3B: W&B Micro-Sweep on CORAL Learning Rate (Addendum Q2)
Runs a 3-5 iteration Bayesian optimization sweep over learning rate `[1e-5, 1e-3]`.

In [ ]:
# Launch sweep and run 5 agents
# !wandb sweep configs/sweep/coral_lr_sweep.yaml
# !wandb agent <SWEEP_ID> --count 5

### Step 3C: EfficientNet-B3 Stretch Run (Addendum Q4)
Single run with the winning CORAL configuration for the headline comparison table.

In [ ]:
# Run B3 benchmark
# !python scripts/train.py model=efficientnet_b3 loss=coral data.batch_size=16

---
## 🌡️ Phase 4: Temperature Scaling Calibration & Reliability Diagrams

In [ ]:
!python scripts/calibrate.py model=efficientnet_b0 loss=coral inference.checkpoint_path=checkpoints/coral_best.pth

---
## 🔬 Phase 5: Grad-CAM Explainability Gallery Generation

In [ ]:
# Generates Grad-CAM overlays for sample fundus images
!python scripts/predict.py --image data/aptos2019/train_images/sample.png --checkpoint checkpoints/coral_best.pth --use-tta

---
## 🌐 Phase 6: Messidor-2 Zero-Shot Domain Shift Evaluation (with TTA)

In [ ]:
!python scripts/evaluate.py data=messidor2 model=efficientnet_b0 loss=coral inference.checkpoint_path=checkpoints/coral_best.pth inference.tta=true